# IMU 辅助动态接收机评审

这本 notebook 不是另一份完整汇报，而是给 `issue_04` 报告补充几张解释型图片。

它回答两个问题：

1. 上一次“纯接收机 / 未耦合”阶段到底停在哪一层。
2. 当前 IMU/INS 接入后，前端、INS 和动态导航的中间量长什么样。

固定口径：

- 旧基线取 `issue_03` 的纯接收机结论和 `22.5 GHz` 代表频点。
- 新链路取 `issue_04` 的代表场景 `h80km_v8.45kms_18p5GHz`。
- 这里做的是“角色与现象对照”，不是同频同工况的严格标定。


In [1]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches

from nav_ka.studies.issue_03_textbook_correction import run_textbook_single_channel_for_frequency
from nav_ka.studies.issue_04_imu_aided import ImuProfileConfig, TrajectoryScenarioConfig, mechanize_imu_profile

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ISSUE03_ROOT = ROOT / 'archive' / 'research' / 'corrections' / 'issue_03_textbook_full_correction'
ISSUE04_ROOT = ROOT / 'archive' / 'research' / 'corrections' / 'issue_04_imu_aided'
FIG_DIR = ISSUE04_ROOT / 'figures' / 'notebook'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.family'] = ['Hiragino Sans GB', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9


def load_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open('r', newline='', encoding='utf-8') as fp:
        return list(csv.DictReader(fp))


def save_fig(fig: plt.Figure, name: str) -> Path:
    path = FIG_DIR / name
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return path


def compute_attitude_error_deg(reference_dcm: np.ndarray, mechanized_dcm: np.ndarray) -> np.ndarray:
    errors_deg = np.empty(len(reference_dcm), dtype=float)
    for idx in range(len(reference_dcm)):
        rot_err = reference_dcm[idx].T @ mechanized_dcm[idx]
        trace_val = np.clip((np.trace(rot_err) - 1.0) * 0.5, -1.0, 1.0)
        errors_deg[idx] = math.degrees(math.acos(trace_val))
    return errors_deg


## 1. 载入旧基线和当前代表场景

这里直接调用仓库现有核心代码，不在 notebook 内复制算法。


In [2]:
issue03_summary = json.loads((ISSUE03_ROOT / 'corrected_fullstack' / 'single_channel' / 'summary.json').read_text(encoding='utf-8'))
issue03_row = next(row for row in issue03_summary['receiver_runs'] if abs(float(row['frequency_hz']) - 22.5e9) <= 1.0)

scenario = TrajectoryScenarioConfig(height_km=80.0, speed_kms=8.45, carrier_frequency_hz=18.5e9)
reference, measured_imu, mechanization, motion_profile, aiding_profile = mechanize_imu_profile(scenario, ImuProfileConfig())

pure_run = run_textbook_single_channel_for_frequency(22.5e9, truth_free_runtime=True, nav_data_enabled=True)
imu_run = run_textbook_single_channel_for_frequency(
    scenario.carrier_frequency_hz,
    truth_free_runtime=True,
    nav_data_enabled=True,
    motion_profile=motion_profile,
    aiding_profile=aiding_profile,
)

state_rows = [
    row
    for row in load_csv_rows(ISSUE04_ROOT / 'corrected_fullstack' / scenario.label / 'dynamic_state_history.csv')
    if row['mode'] == 'ekf_pr_doppler' and row['vel_err_3d_mps'] not in {'', 'nan', 'None'}
]

print({
    'issue03_baseline_label': issue03_row['label'],
    'issue03_tau_rmse_ns': issue03_row['tau_rmse_ns'],
    'issue03_fd_rmse_hz': issue03_row['fd_rmse_hz'],
    'issue04_case': scenario.label,
    'issue04_tau_rmse_ns': imu_run.metrics['tau_rmse_ns'],
    'issue04_fd_rmse_hz': imu_run.metrics['fd_rmse_hz'],
    'num_state_rows': len(state_rows),
})



[接收机时间轴构造]
  - 真实 WKB 时长 T_real = 21.908328000 s
  - 接收机采样率 fs = 500000.000 Hz
  - 离散化后总采样点 N = 10954164
  - 接收机实际使用时长 T_used = 21.908328000 s
  - 离散化误差 = +1.776e-14 s

[步骤 B] 构造 Ka 22.5 GHz 基础 PN/BPSK 发射模型
  - λ = 1.3324 cm
  - 采样率 fs = 500.000 kHz
  - 码率 chip_rate = 50.000 kcps
  - 每码片采样点 = 10
  - 总时长 = 21.908328 s（完全继承真实 WKB 时间范围）



[步骤 C+] 捕获物理诊断
  - 真值参考时刻 = 5.000 ms
  - 真值频偏 = 47.992 kHz
  - 真值码时延(模一码周期) = 346.020 us
  - 粗频偏误差 = +7.957 Hz
  - 粗码相位误差 = +179.528 ns (+0.090 samples, +0.0090 chips)
  - 主峰/次峰比 = 1.000 (0.001 dB)
  - 频偏搜索步进 = 2000.0 Hz
  - 码相位搜索步进 = 1 samples
    top1: fd = 48.000 kHz, tau = 346.000 us, metric = 72.532 dB, fd_err = +8.0 Hz, tau_err = -20.5 ns
    top2: fd = 48.000 kHz, tau = 348.000 us, metric = 72.531 dB, fd_err = +8.0 Hz, tau_err = +1979.5 ns
    top3: fd = 48.000 kHz, tau = 344.000 us, metric = 71.876 dB, fd_err = +8.0 Hz, tau_err = -2020.5 ns
    top4: fd = 48.000 kHz, tau = 350.000 us, metric = 71.692 dB, fd_err = +8.0 Hz, tau_err = +3979.5 ns
    top5: fd = 48.000 kHz, tau = 342.000 us, metric = 70.893 dB, fd_err = +8.0 Hz, tau_err = -4020.5 ns



[步骤 D+] 跟踪物理诊断
  - 预测输入 C/N0 范围 = [8.032, 44.804] dB-Hz
  - 预测 1 ms 后相关 SNR 范围 = [-21.968, 14.804] dB
  - 实测 Prompt 后相关 SNR 范围 = [-34.133, 20.128] dB
  - Prompt/(Early+Late)/2 中位数 = 1.300
  - 载波锁定指标范围 = [-1.000, 1.000], 弱锁定占比 = 50.78%
  - PLL 判别器超线性区占比 = 65.50% (阈值 |e_pll| > 0.35 rad)
  - DLL 判别器超线性区占比 = 18.39% (阈值 |e_dll| > 0.20)
  - Prompt SNR < 0.0 dB 占比 = 12.82%
  - FLL 辅助占比 = 58.02%
  - DLL 冻结占比 = 57.51%
  - PLL 冻结占比 = 7.48%
  - PLL 积分器夹限次数 = 0
  - PLL 频率命令夹限次数 = 0
  - 真值多普勒范围 = [8.567, 48.000] kHz, 最大变化率 = 3376.897 Hz/s
  - 真值码时延变化率范围 = [3999.081, 4000.795] ns/s
  - PLL 积分器工作范围 = [-29.063, -0.003] kHz
  - PLL 频率命令范围 = [18.657, 48.241] kHz
  - FLL 误差范围 = [-499.941, 499.966] Hz
  - 首次持续失锁时刻 = 未检测到 (判据窗口 20 blocks)
    前段: tau_RMSE = 533.048 ns, fd_RMSE = 147.706 Hz, Prompt_SNR_med = 17.217 dB, PLL_RMS = 0.5700 rad, DLL_RMS = 0.0987, loss_frac = 1.94%
    中段: tau_RMSE = 546.507 ns, fd_RMSE = 125.602 Hz, Prompt_SNR_med = 16.321 dB, PLL_RMS = 0.5587 rad, DLL_RMS = 0.1066, loss_frac =


[步骤 C+] 捕获物理诊断
  - 真值参考时刻 = 5.000 ms
  - 真值频偏 = 47.992 kHz
  - 真值码时延(模一码周期) = 346.020 us
  - 粗频偏误差 = +7.718 Hz
  - 粗码相位误差 = +179.525 ns (+0.090 samples, +0.0090 chips)
  - 主峰/次峰比 = 1.013 (0.054 dB)
  - 频偏搜索步进 = 2000.0 Hz
  - 码相位搜索步进 = 1 samples
    top1: fd = 48.000 kHz, tau = 346.000 us, metric = 72.160 dB, fd_err = +7.7 Hz, tau_err = -20.5 ns
    top2: fd = 48.000 kHz, tau = 348.000 us, metric = 72.106 dB, fd_err = +7.7 Hz, tau_err = +1979.5 ns
    top3: fd = 48.000 kHz, tau = 344.000 us, metric = 71.567 dB, fd_err = +7.7 Hz, tau_err = -2020.5 ns
    top4: fd = 48.000 kHz, tau = 350.000 us, metric = 71.254 dB, fd_err = +7.7 Hz, tau_err = +3979.5 ns
    top5: fd = 48.000 kHz, tau = 342.000 us, metric = 70.647 dB, fd_err = +7.7 Hz, tau_err = -4020.5 ns



[步骤 D+] 跟踪物理诊断
  - 预测输入 C/N0 范围 = [-25.245, 44.707] dB-Hz
  - 预测 1 ms 后相关 SNR 范围 = [-55.245, 14.707] dB
  - 实测 Prompt 后相关 SNR 范围 = [-46.010, 18.981] dB
  - Prompt/(Early+Late)/2 中位数 = 1.004
  - 载波锁定指标范围 = [-1.000, 1.000], 弱锁定占比 = 54.56%
  - PLL 判别器超线性区占比 = 77.48% (阈值 |e_pll| > 0.35 rad)
  - DLL 判别器超线性区占比 = 55.30% (阈值 |e_dll| > 0.20)
  - Prompt SNR < 0.0 dB 占比 = 51.47%
  - FLL 辅助占比 = 60.92%
  - DLL 冻结占比 = 81.64%
  - PLL 冻结占比 = 30.75%
  - PLL 积分器夹限次数 = 0
  - PLL 频率命令夹限次数 = 1732
  - 真值多普勒范围 = [76.413, 94.782] kHz, 最大变化率 = 4381.267 Hz/s
  - 真值码时延变化率范围 = [-5123.328, -4130.740] ns/s
  - PLL 积分器工作范围 = [-0.190, 0.619] kHz
  - PLL 频率命令范围 = [22.113, 136.274] kHz
  - FLL 误差范围 = [-499.965, 499.981] Hz
  - 首次持续失锁时刻 = 未检测到 (判据窗口 20 blocks)
    前段: tau_RMSE = 35930.655 ns, fd_RMSE = 34989.678 Hz, Prompt_SNR_med = 0.686 dB, PLL_RMS = 0.6153 rad, DLL_RMS = 0.2475, loss_frac = 51.29%
    中段: tau_RMSE = 21799.671 ns, fd_RMSE = 56921.379 Hz, Prompt_SNR_med = 0.198 dB, PLL_RMS = 0.6189 rad, DLL_RMS = 0.24

## 2. 旧纯接收机与当前 IMU 耦合链路的角色差异

这张图的目的不是比数值，而是把链路边界和新增角色画清楚。


In [3]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')

left_boxes = [
    (0.04, 0.62, 0.17, 0.18, '信号与信道', '#d9ead3'),
    (0.27, 0.62, 0.19, 0.18, '纯接收机\n(捕获 / 跟踪)', '#fff2cc'),
    (0.53, 0.62, 0.18, 0.18, '接收机直接测量量\n与标准观测量', '#cfe2f3'),
]
right_boxes = [
    (0.04, 0.18, 0.17, 0.18, '信号与信道', '#d9ead3'),
    (0.27, 0.18, 0.16, 0.18, '惯性测量序列', '#fce5cd'),
    (0.47, 0.18, 0.18, 0.18, '惯性导航解算', '#c9daf8'),
    (0.69, 0.18, 0.14, 0.18, '跟踪辅助量\n与运动信息', '#f4cccc'),
    (0.86, 0.18, 0.10, 0.18, '接收机与\nWLS / EKF', '#ead1dc'),
]

for x, y, w, h, text, color in left_boxes + right_boxes:
    box = patches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.012,rounding_size=0.02', facecolor=color, edgecolor='#3c3c3c', lw=1.2)
    ax.add_patch(box)
    ax.text(x + w / 2.0, y + h / 2.0, text, ha='center', va='center', fontsize=10)

for start, end in [
    ((0.21, 0.71), (0.27, 0.71)),
    ((0.46, 0.71), (0.53, 0.71)),
    ((0.21, 0.27), (0.27, 0.27)),
    ((0.43, 0.27), (0.47, 0.27)),
    ((0.65, 0.27), (0.69, 0.27)),
    ((0.83, 0.27), (0.86, 0.27)),
]:
    ax.annotate('', xy=end, xytext=start, arrowprops=dict(arrowstyle='->', lw=1.8, color='#4d4d4d'))

ax.text(0.50, 0.93, '上一轮纯接收机方案与本轮惯性辅助方案的链路对比', ha='center', va='center', fontsize=14, fontweight='bold')
ax.text(0.50, 0.84, '上一轮方案的链路终点停留在接收机直接测量量和标准观测量。', ha='center', va='center', fontsize=10)
ax.text(0.50, 0.08, '本轮方案在此基础上增加了惯性导航提供的运动信息、跟踪辅助量和预测状态。', ha='center', va='center', fontsize=10)
ax.text(0.02, 0.84, '上一轮\n纯接收机方案', fontsize=11, fontweight='bold', ha='left')
ax.text(0.02, 0.40, '本轮\n惯性辅助方案', fontsize=11, fontweight='bold', ha='left')

path = save_fig(fig, 'receiver_chain_role_comparison.png')
print(path)


/Users/guozehao/Documents/ka-Nav/nav_ka_github/archive/research/corrections/issue_04_imu_aided/figures/notebook/receiver_chain_role_comparison.png


## 3. 纯接收机基线与当前 IMU 辅助前端的细节对照

这里放在一起比较的是接收机层能直接看到的量：`tau/fd` 恢复误差、后相关 SNR 和锁定指标。


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex='col')

pure_t = np.asarray(pure_run.trk_result['t'], dtype=float)
pure_tau_err_ns = (np.asarray(pure_run.trk_result['tau_est_s'], dtype=float) - np.asarray(pure_run.trk_result['tau_true_s'], dtype=float)) * 1e9
pure_fd_err_hz = np.asarray(pure_run.trk_result['doppler_hz'], dtype=float) - np.asarray(pure_run.trk_result['fd_true_hz'], dtype=float)

imu_t = np.asarray(imu_run.trk_result['t'], dtype=float)
imu_tau_err_ns = (np.asarray(imu_run.trk_result['tau_est_s'], dtype=float) - np.asarray(imu_run.trk_result['tau_true_s'], dtype=float)) * 1e9
imu_fd_err_hz = np.asarray(imu_run.trk_result['doppler_hz'], dtype=float) - np.asarray(imu_run.trk_result['fd_true_hz'], dtype=float)

axes[0, 0].plot(pure_t, pure_tau_err_ns, color='#4f81bd', lw=1.1)
axes[0, 0].set_title('上一轮纯接收机方案：22.5 GHz 码时延误差')
axes[0, 0].set_ylabel('ns')
axes[0, 0].grid(alpha=0.25)

axes[0, 1].plot(imu_t, imu_tau_err_ns, color='#c0504d', lw=1.1)
axes[0, 1].set_title('本轮惯性辅助方案：80 km、8.45 km/s、18.5 GHz 的码时延误差')
axes[0, 1].grid(alpha=0.25)

axes[1, 0].plot(pure_t, np.asarray(pure_run.trk_result['post_corr_snr_db'], dtype=float), color='#4f81bd', lw=1.0, label='后相关信噪比')
ax10r = axes[1, 0].twinx()
ax10r.plot(pure_t, np.asarray(pure_run.trk_result['carrier_lock_metric'], dtype=float), color='#9bbb59', lw=1.0, label='锁定指标')
axes[1, 0].set_title('上一轮纯接收机方案：信噪比与锁定情况')
axes[1, 0].set_ylabel('dB')
ax10r.set_ylabel('指标值')
axes[1, 0].set_xlabel('时间（s）')
axes[1, 0].grid(alpha=0.25)

axes[1, 1].plot(imu_t, np.asarray(imu_run.trk_result['post_corr_snr_db'], dtype=float), color='#c0504d', lw=1.0, label='后相关信噪比')
ax11r = axes[1, 1].twinx()
ax11r.plot(imu_t, np.asarray(imu_run.trk_result['carrier_lock_metric'], dtype=float), color='#9bbb59', lw=1.0, label='锁定指标')
axes[1, 1].set_title('本轮惯性辅助方案：信噪比与锁定情况')
axes[1, 1].set_xlabel('时间（s）')
axes[1, 1].grid(alpha=0.25)

fig.suptitle('接收机层细节对照：纯接收机方案与惯性辅助方案', fontsize=14, fontweight='bold')
path = save_fig(fig, 'pure_receiver_vs_imu_aided_tracking.png')
print(path)
print({'issue03_tau_rmse_ns': pure_run.metrics['tau_rmse_ns'], 'issue04_tau_rmse_ns': imu_run.metrics['tau_rmse_ns'], 'issue03_fd_rmse_hz': pure_run.metrics['fd_rmse_hz'], 'issue04_fd_rmse_hz': imu_run.metrics['fd_rmse_hz']})


/Users/guozehao/Documents/ka-Nav/nav_ka_github/archive/research/corrections/issue_04_imu_aided/figures/notebook/pure_receiver_vs_imu_aided_tracking.png
{'issue03_tau_rmse_ns': 8784.13918967316, 'issue04_tau_rmse_ns': 34704.58635251874, 'issue03_fd_rmse_hz': 3166.2904656226565, 'issue04_fd_rmse_hz': 41056.66941366851}


## 4. IMU/INS 中间量和导航细节

下面两张图更靠近这轮新增内容本身：一张看 INS mechanization，一张看导航层可用性与误差。


In [5]:
pos_err_m = np.linalg.norm(mechanization.position_ecef_m - reference.position_ecef_m, axis=1)
vel_err_mps = np.linalg.norm(mechanization.velocity_ecef_mps - reference.velocity_ecef_mps, axis=1)
att_err_deg = compute_attitude_error_deg(reference.attitude_dcm_bn, mechanization.attitude_dcm_bn)

fig, axes = plt.subplots(3, 1, figsize=(13.5, 9), sharex=True)
axes[0].plot(measured_imu.t_s, measured_imu.gyro_body_radps[:, 0], label='角速度 x', lw=1.0)
axes[0].plot(measured_imu.t_s, measured_imu.gyro_body_radps[:, 1], label='角速度 y', lw=1.0)
axes[0].plot(measured_imu.t_s, measured_imu.gyro_body_radps[:, 2], label='角速度 z', lw=1.1)
axes[0].set_ylabel('rad/s')
axes[0].set_title('角速度测量序列')
axes[0].legend(ncol=3, fontsize=8, loc='upper right')
axes[0].grid(alpha=0.25)

axes[1].plot(measured_imu.t_s, measured_imu.accel_body_mps2[:, 0], label='比力 x', lw=1.0)
axes[1].plot(measured_imu.t_s, measured_imu.accel_body_mps2[:, 1], label='比力 y', lw=1.0)
axes[1].plot(measured_imu.t_s, measured_imu.accel_body_mps2[:, 2], label='比力 z', lw=1.0)
axes[1].set_ylabel('m/s²')
axes[1].set_title('机体系比力测量序列')
axes[1].legend(ncol=3, fontsize=8, loc='upper right')
axes[1].grid(alpha=0.25)

axes[2].plot(reference.t_s, pos_err_m, color='#c0504d', lw=1.2, label='位置误差')
axes[2].plot(reference.t_s, vel_err_mps * 12.0, color='#4f81bd', lw=1.1, label='速度误差 ×12')
ax2r = axes[2].twinx()
ax2r.plot(reference.t_s, att_err_deg, color='#9bbb59', lw=1.1, label='姿态误差')
axes[2].set_ylabel('m / 缩放后的 m/s')
ax2r.set_ylabel('°')
axes[2].set_xlabel('时间（s）')
axes[2].set_title('惯性导航误差包络')
axes[2].grid(alpha=0.25)
lines1, labels1 = axes[2].get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
axes[2].legend(lines1 + lines2, labels1 + labels2, ncol=3, fontsize=8, loc='upper right')

fig.suptitle('典型场景下的惯性测量量与惯性导航细节', fontsize=14, fontweight='bold')
path = save_fig(fig, 'imu_ins_error_detail.png')
print(path)
print({'ins_mean_position_error_m': float(np.mean(pos_err_m)), 'ins_mean_velocity_error_mps': float(np.mean(vel_err_mps)), 'ins_mean_attitude_error_deg': float(np.mean(att_err_deg))})


/Users/guozehao/Documents/ka-Nav/nav_ka_github/archive/research/corrections/issue_04_imu_aided/figures/notebook/imu_ins_error_detail.png
{'ins_mean_position_error_m': 1.5816197133435423, 'ins_mean_velocity_error_mps': 0.14594286658002367, 'ins_mean_attitude_error_deg': 0.0019343804286944958}


In [6]:
state_t_s = np.asarray([float(row['t_s']) for row in state_rows], dtype=float)
num_valid_sats = np.asarray([float(row['num_valid_sats']) for row in state_rows], dtype=float)
prediction_only = np.asarray([1.0 if row['prediction_only'].lower() == 'true' else 0.0 for row in state_rows], dtype=float)
pos_err = np.asarray([float(row['pos_err_3d_m']) for row in state_rows], dtype=float)
vel_err = np.asarray([float(row['vel_err_3d_mps']) for row in state_rows], dtype=float)
innov_pr = np.asarray([float(row['innovation_rms_pr_m']) if row['innovation_rms_pr_m'] not in {'', 'nan', 'None'} else np.nan for row in state_rows], dtype=float)
innov_rr = np.asarray([float(row['innovation_rms_rr_mps']) if row['innovation_rms_rr_mps'] not in {'', 'nan', 'None'} else np.nan for row in state_rows], dtype=float)

fig, axes = plt.subplots(3, 1, figsize=(13.5, 9), sharex=True)
axes[0].plot(imu_run.trk_result['t'], np.asarray(imu_run.trk_result['imu_code_aiding_rate_chips_per_s'], dtype=float), lw=1.1, color='#4f81bd', label='码率辅助量')
ax0r = axes[0].twinx()
ax0r.plot(imu_run.trk_result['t'], np.asarray(imu_run.trk_result['imu_carrier_aiding_rate_hz_per_s'], dtype=float), lw=1.1, color='#c0504d', label='载波频率辅助量')
axes[0].set_ylabel('chips/s')
ax0r.set_ylabel('Hz/s')
axes[0].set_title('跟踪辅助量时序')
axes[0].grid(alpha=0.25)
lines1, labels1 = axes[0].get_legend_handles_labels()
lines2, labels2 = ax0r.get_legend_handles_labels()
axes[0].legend(lines1 + lines2, labels1 + labels2, ncol=2, fontsize=8, loc='upper right')

axes[1].step(state_t_s, num_valid_sats, where='mid', color='#9bbb59', lw=1.2, label='有效卫星数')
axes[1].step(state_t_s, prediction_only * np.max(num_valid_sats), where='mid', color='#8064a2', lw=1.1, label='纯预测历元')
axes[1].set_ylabel('数量 / 标记')
axes[1].set_title('观测更新情况')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(alpha=0.25)

axes[2].plot(state_t_s, pos_err, color='#c0504d', lw=1.2, label='位置误差')
axes[2].plot(state_t_s, vel_err * 20.0, color='#4f81bd', lw=1.1, label='速度误差 ×20')
ax2r = axes[2].twinx()
ax2r.plot(state_t_s, innov_pr, color='#f79646', lw=1.0, label='伪距创新均方根')
ax2r.plot(state_t_s, innov_rr * 60.0, color='#7f7f7f', lw=1.0, label='伪距率创新均方根 ×60')
axes[2].set_ylabel('m / 缩放后的 m/s')
ax2r.set_ylabel('创新量')
axes[2].set_xlabel('时间（s）')
axes[2].set_title('导航误差与创新量细节')
axes[2].grid(alpha=0.25)
lines1, labels1 = axes[2].get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
axes[2].legend(lines1 + lines2, labels1 + labels2, ncol=4, fontsize=8, loc='upper right')

fig.suptitle('典型场景下的导航细节', fontsize=14, fontweight='bold')
path = save_fig(fig, 'imu_navigation_error_detail.png')
print(path)
print({'prediction_only_epochs': float(np.sum(prediction_only)), 'mean_valid_sats': float(np.mean(num_valid_sats)), 'mean_position_error_m': float(np.nanmean(pos_err))})


/Users/guozehao/Documents/ka-Nav/nav_ka_github/archive/research/corrections/issue_04_imu_aided/figures/notebook/imu_navigation_error_detail.png
{'prediction_only_epochs': 0.0, 'mean_valid_sats': 1.3333333333333333, 'mean_position_error_m': 66.1311648127854}
